# 实验3：基于统计语言模型的中文句子缺词补全

> **课程**：自然语言处理  
> **实验对象**：本科生  
> **实验方式**：上机独立实践  
> **实验环境**：使用前期已经配置好的 Python 环境，可采用 Jupyter Notebook 或 Python 脚本完成  
> **实验主题**：实现统计语言模型，并将其用于中文句子缺词补全


本实验分为两个任务：

- **任务一：统计语言模型的理论到实现。**通过小型语料掌握 N-gram 统计、概率计算和平滑方法。
- **任务二：基于统计语言模型的项目实操。**围绕“中文句子缺词补全”项目，独立完成数据读取、模型构建、候选评分、结果输出和分析。


---


# 一、实验目的

完成实验后，应能够：

1. 根据文本构建 Unigram、Bigram 和 Trigram 统计结果。
2. 根据计数计算条件概率和句子概率。
3. 理解零概率问题，实现加一、加 K 和插值平滑。
4. 使用统计语言模型比较不同候选句子的得分。
5. 分析模型阶数、平滑参数和训练语料对实验结果的影响。
6. 独立完成一个基于统计语言模型的中文句子缺词补全项目。


# 二、实验安排与准备

| 阶段 | 子任务 | 主要产出 |
|---|---|---|
| 任务一 | 1. N-gram统计 | 三种阶数的计数、词表和计数变化 |
| 任务一 | 2. 条件概率与句子概率 | 指定查询、逐项概率和总得分 |
| 任务一 | 3. 平滑 | 加一、加K、插值与归一化检查 |
| 任务一 | 4. 模型比较 | 同一批句子的得分对照与解释 |
| 任务一 | 5. 缺词补全原理 | 完整句评分与右侧信息分析 |
| 任务二 | 1—3. 数据、模型与功能 | 独立实现批量候选评分与排序 |
| 任务二 | 4—5. 验证与测试 | 固定配置、预测文件和评价 |
| 任务二 | 6. 挑战题与独立编题 | 六道挑战题及至少六道自编题分析 |



## 1. 文件组织

请先完整解压实验材料，保留如下结构：

```text
实验3_统计语言模型/
├── NLP实验3_统计语言模型_实验指导.ipynb
├── README_使用说明.md
└── data/
    ├── train.txt
    ├── dev.csv
    ├── test.csv
    └── challenge.csv
```

使用前期已配置好的 Python 环境完成实验。

建议从 Notebook 所在目录启动 Jupyter。以下单元格仅检查环境及文件位置，不读取或处理项目数据。


In [ ]:
from pathlib import Path
import sys

DATA_DIR = Path("data")
print("Python 版本：", sys.version.split()[0])
print("当前目录：", Path.cwd())
print("数据目录：", DATA_DIR.resolve())
for name in ("train.txt", "dev.csv", "test.csv", "challenge.csv"):
    print(name, "已找到" if (DATA_DIR / name).is_file() else "未找到，请检查解压位置或工作目录")


## 2. 全实验统一约定

- 输入已经按空格分词；每行一句，不跨句构造 N-gram。
- N 元模型每句句首补充 $N-1$ 个 `<s>`，句末补充一个 `</s>`。
- `<s>` 只作为上下文，不作为预测词；Unigram 不补句首标记。
- 预测词表 $V$ 包含训练语料中的普通词语、`</s>` 和 `<unk>`，不含 `<s>` 和空位 `____`。
- 仅根据相应任务的训练文本建词表；任务一与任务二使用各自的训练语料，不能混合统计。
- 训练词全部保留；评分时将词表外普通词映射为 `<unk>`，上下文中的词表外普通词也按同一规则处理。特殊句首标记保持为 `<s>`。
- 计数是按词元而非字符串子串进行的。“发布会”是一个完整词，不计入“发布”。
- 各阶模型均只统计实际预测词及句末标记的事件；不要把补齐的句首标记统计为预测目标。
- 句子评分统一使用自然对数；包含句末预测，不包含预测 `<s>` 的步骤。有限得分越大（通常越接近0），表示当前模型赋予的概率越高。
- 未平滑模型中，“已见历史、未见后继”的条件概率为0；“未见历史”则无法由极大似然估计条件分布。二者必须区分。

**实现提醒：**正K平滑时，保留但没有训练计数的 `<unk>` 也参与分母中的词表大小。每种阶数、每个插值分量必须使用同一预测词表。

---


# 三、任务一：统计语言模型的理论到实现

本任务使用少量、便于观察的文本，依次完成：

**文本读取 → N-gram统计 → 概率估计 → 平滑处理 → 模型比较。**


## 子任务1：构建 N-gram 统计结果

### 1. 实验材料

使用以下六个已经分词的句子。每行是一个独立句子，空格表示词语边界。


In [ ]:
toy_text = """
学校 今天 发布 通知
学校 明天 发布 通知
学生 今天 提交 报告
学生 明天 提交 报告
教师 今天 发布 作业
教师 明天 提交 作业"""

# 实验文本；请完成读取、分词与词表构建。
print(toy_text)


### 2. 实验内容

分别统计：

- **Unigram：**单个词语出现的次数；
- **Bigram：**连续两个词语出现的次数；
- **Trigram：**连续三个词语出现的次数。

以下示例仅说明连续片段，暂不列特殊标记；正式模型应遵循前述边界约定。

```text
句子：学校 今天 发布 通知

Unigram：学校；今天；发布；通知
Bigram：学校 今天；今天 发布；发布 通知
Trigram：学校 今天 发布；今天 发布 通知
```

### 3. 关键实现提示

可以使用 Python 字典或 `Counter` 保存计数。下面只展示 Bigram 计数的基本写法，其中 `sentences` 是按句组织的词语列表。

`history_counts[history]` 统计这个历史对应的预测事件数；它不是含义不明的“该词在任何位置的次数”。建议将通用 N-gram 历史存为元组，以便统一空历史、单词历史和双词历史的实现。


In [ ]:
from collections import Counter

def count_bigram_demo(sentences):
    """统计二元事件与对应历史。"""
    bigram_counts = Counter()
    history_counts = Counter()
    for words in sentences:
        sequence = ["<s>"] + words + ["</s>"]
        for history, word in zip(sequence[:-1], sequence[1:]):
            bigram_counts[(history, word)] += 1
            history_counts[history] += 1
    return bigram_counts, history_counts


请在此基础上独立完成 Unigram 和 Trigram 的统计。Trigram 每句补两个 `<s>`，从第一个普通词开始预测；不要计入“`<s>`之后预测另一个`<s>`”这样的事件。

### 4. 需要完成

1. 将 `toy_text` 按行读取并转换为按句组织的词语列表。
2. 建立普通词表和预测词表，说明二者的区别。
3. 输出三种阶数的统计结果，明确是否含特殊标记。
4. 查询指定词语或词组的出现次数。
5. 增加或修改一句文本，观察计数变化；后续比较恢复使用原始六句语料，并保留修改前后的记录。

| 统计对象 | 示例 | 程序统计值 |
|---|---|---:|
| 单个词 | 发布 | 待填写 |
| 两个连续词 | 发布 通知 | 待填写 |
| 三个连续词 | 今天 发布 通知 | 待填写 |

**验收材料：**按句组织的词语列表、词表、三种阶数的完整计数、指定查询及修改记录。可把较长的计数表另存文件，并在此处给出文件名和关键条目。


In [ ]:
# 实践区：任务一 / 子任务1
# TODO：将 toy_text 转为 toy_sentences，建立 toy_vocab。
# TODO：调用 Bigram 示例；自行实现 Unigram、Trigram 统计。
# TODO：输出计数与查询结果，记录修改语料前后的变化。
# TODO：后续实验恢复原始六句语料。


### 分析区：计数与边界

- 普通词表大小、预测词表大小：待填写。
- 三种阶数的预测事件总数及其核对方法：待填写。
- 三个指定计数：待填写。
- 增改的句子与计数变化：待填写。
- 若把多句直接拼接，会出现哪些不应存在的 N-gram：待填写。

---


## 子任务2：计算条件概率和句子概率

### 1. 概率估计

对于 Unigram：

$$
P(w)=\frac{C(w)}{\sum_{v\in V}C(v)}
$$

这里的词计数包含句末 `</s>`，不包含句首 `<s>`。

对于 Bigram：

$$
P(w_i\mid w_{i-1})=
\frac{C(w_{i-1},w_i)}{\sum_{w\in V}C(w_{i-1},w)}
$$

对于 Trigram：

$$
P(w_i\mid w_{i-2},w_{i-1})=
\frac{C(w_{i-2},w_{i-1},w_i)}{\sum_{w\in V}C(w_{i-2},w_{i-1},w)}
$$

分母表示相应历史之后出现的所有预测事件的总次数，记作 $C(h)$。对 Unigram 可把历史看作空元组，其历史总数是所有预测事件的总数。

### 2. 句子概率与对数得分

设原句含 $m$ 个词，则还要预测一次句末标记：

$$
P(S)=\prod_{i=1}^{m+1}P(w_i\mid h_i),\qquad w_{m+1}=\texttt{</s>}
$$

程序中建议使用对数概率相加：

$$
\log P(S)=\sum_{i=1}^{m+1}\log P(w_i\mid h_i)
$$

**关键实现提示：**

- 每一步记录历史、预测词、分子计数、分母计数、概率和对数概率。
- 条件概率为0时，对数概率记为负无穷 `-inf`；不要直接调用 `math.log(0)`。
- 未见历史导致无法估计时，应记录该状态及具体历史。可以用 `None` 或单独状态字段表示，不能与已定义但为0的概率混淆。
- 只有每一步概率都已定义时，才讨论该模型给出的句子概率；如果出现无法估计的步骤，单独标记无法评分。
- 对数相加避免大量小概率连乘的数值下溢，但不会消除模型本身的零概率问题。

### 3. 需要完成

1. 查询并输出以下概率，附分子和分母：
   - “今天”之后出现“发布”的概率；
   - “发布”之后出现“通知”的概率；
   - “学校 今天”之后出现“发布”的概率。
2. 分别使用 Unigram、Bigram、Trigram 计算下面两个句子的得分。
3. 至少选择一句话，输出每一步条件概率及对数概率，包含句末预测。
4. 说明最终得分如何由逐项概率得到。

```text
学校 今天 发布 通知
学校 今天 通知 发布
```

这两个句子的词袋相同、词序不同。


In [ ]:
# 实践区：任务一 / 子任务2
# TODO：实现未平滑的条件概率查询，区分零概率与未见历史。
# TODO：输出三个指定查询的计数、分母、概率。
# TODO：实现含句末预测的对数评分，并输出逐项记录。
# TODO：使用 Unigram、Bigram、Trigram 比较两个句子。


### 分析区：条件概率与得分

| 查询 | 联合计数 | 历史预测事件数 | 条件概率 |
|---|---:|---:|---:|
| 今天 → 发布 | 待填写 | 待填写 | 待填写 |
| 发布 → 通知 | 待填写 | 待填写 | 待填写 |
| 学校 今天 → 发布 | 待填写 | 待填写 | 待填写 |

**逐项得分表**（按实际步数增行）：

| 步骤 | 历史 | 预测词 | 条件概率 | 对数概率 | 状态 |
|---:|---|---|---:|---:|---|
| 1 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| … | … | … | … | … | … |
| 句末 | 待填写 | `</s>` | 待填写 | 待填写 | 待填写 |

模型与总得分：待填写。请保留实际计算精度，展示时可以四舍五入。

请回答：

1. Unigram 能否区分这两个句子的词序？为什么？
2. Bigram、Trigram 的得分或可评分状态是否不同？
3. 某个词组没有出现过时，会怎样影响句子概率？历史未见与后继未见有什么不同？
4. 为什么长句不宜直接使用大量小概率连乘？对数变换能否解决零概率？

回答：待填写。

---


## 子任务3：实现平滑方法

### 1. 观察零概率问题

先使用以下没有作为完整句子出现在训练语料中的句子：

```text
学校 今天 提交 报告
```

**观察提示：整句未见不等于一定有零概率。**这句话的所有 Bigram 都在训练中出现过，因此未平滑 Bigram 可以给出非零概率；Trigram 中的“学校 今天 → 提交”未出现。请分别运行两种阶数，按实际结果定位并解释，不要预设两者都会产生零概率。

为进一步观察 Bigram 的零概率，可增补：

```text
学校 今天 通知 报告
```

不要只记录“句子概率为0”，还要列出具体哪一步条件概率为0、哪一步历史未见。检查逐项事件，即使前一步已为0，也继续收集后续步骤的诊断信息。

### 2. 实现加一与加 K 平滑

统一写为：

$$
P_{+k}(w\mid h)=\frac{C(h,w)+k}{C(h)+k|V|}
$$

其中：

- $C(h,w)$：历史与后继词共同出现的次数；
- $C(h)$：历史对应的预测事件总数；
- $|V|$：预测词表大小；
- $k=1$：加一平滑；正 K 平滑要求 $k>0$。

比较以下设置：

```text
不平滑
k = 0.1
k = 0.5
k = 1
```

当历史未见且 $k>0$ 时，上式仍有定义。对“不平滑”分支单独检查分母；不能把 $k=0,C(h)=0$ 直接代入后当成一个合法概率。

### 3. 实现插值平滑

组合不同阶数模型，在同一预测位置计算：

$$
P_{\mathrm{mix}}(w_i\mid h_i)=
\lambda_1 P_{\mathrm{uni}}(w_i)
+\lambda_2 P_{\mathrm{bi}}(w_i\mid w_{i-1})
+\lambda_3 P_{\mathrm{tri}}(w_i\mid w_{i-2},w_{i-1})
$$

要求：

$$
\lambda_1,\lambda_2,\lambda_3\geq0,\qquad
\lambda_1+\lambda_2+\lambda_3=1
$$

为便于处理未见上下文，先对三个分量分别使用正 K 平滑，再进行插值。三个分量必须使用相同预测词表，并在同一预测词位置对齐；句首按各分量的阶数处理上下文。

**关键实现提示：**先加权得到每一步的混合概率，再取对数并累加。不能把“各模型句子对数得分的加权和”当作本实验要求的概率插值。

### 4. 需要完成

- 输出已见事件和未见事件在不同设置下的概率；
- 比较平滑前后的句子得分；
- 对已见历史和未见历史，遍历整个预测词表，检查正 K 平滑及插值概率之和是否为1；可使用如 $10^{-9}$ 的数值容差；
- 说明为什么平滑可能降低某些高频事件的概率；
- 说明 `<unk>` 没有训练计数时，正 K 平滑如何赋予它非零概率。

可选取以下具体事件：

| 事件类型 | 示例（Bigram） |
|---|---|
| 已见历史、已见后继 | 今天 → 发布 |
| 已见历史、未见后继 | 学校 → 提交 |
| 未见历史 | `<unk>` → 发布 |

**未平滑模型遇到未见历史时，应明确记录“无法估计”，不能把全部为0的结果称为一个归一化条件分布。**


In [ ]:
# 实践区：任务一 / 子任务3
# TODO：比较不平滑与 k=0.1、0.5、1。
# TODO：实现对齐预测位置的三阶概率插值，记录 K 与全部权重。
# TODO：输出具体事件、逐项得分、句子总得分。
# TODO：遍历预测词表，检查已见/未见历史的概率和。


### 分析区：平滑比较

加 K 表列使用的 K：待填写。插值各分量 K 与权重：待填写。

| 事件类型及具体事件 | 未平滑 | 加一 | 加 K | 插值 |
|---|---:|---:|---:|---:|
| 已见历史、已见后继：待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| 已见历史、未见后继：待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| 未见历史：待填写 | 无法估计 | 待填写 | 待填写 | 待填写 |

| 检查对象 | 模型/参数 | 遍历的词表大小 | 概率和 | 是否通过 |
|---|---|---:|---:|---|
| 已见历史 | 待填写 | 待填写 | 待填写 | 待填写 |
| 未见历史 | 待填写 | 待填写 | 待填写 | 待填写 |
| 插值分布 | 待填写 | 待填写 | 待填写 | 待填写 |

1. 两个观察句中，哪些具体事件导致零概率或无法估计：待填写。
2. 不平滑、K=0.1、0.5、1及插值的句子得分对比：待填写。
3. 高频事件的概率为什么可能被降低：待填写。
4. 未见历史经正 K 平滑后得到怎样的分布，是否能区分后继词：待填写。

**选做：绝对折扣平滑。**对已见事件计数做折扣，检查扣除的概率质量是否通过回退或其他明确机制重新分配，并检查归一化。不能仅减去常数而不重新分配概率质量。

---


## 子任务4：比较模型阶数和平滑设置

### 1. 实验材料

```text
句子1：学校 明天 提交 报告
句子2：学生 今天 发布 作业
句子3：教师 今天 提交 报告
句子4：学生 明天 发布 通知
```

这些句子中的词都来自训练语料，但完整组合有所变化。

### 2. 需要完成

1. 在原始六句训练语料上建立不同模型，保持预测词表一致。
2. 对相同句子输出自然对数得分，包含句末预测。
3. 记录未平滑模型的零概率、未见历史等现象，再比较平滑后的结果。
4. 比较不同模型的差异；明确记录 K 值和插值权重。

| 模型 | 平滑设置 | 句子1得分 | 句子2得分 | 句子3得分 | 句子4得分 |
|---|---|---:|---:|---:|---:|
| Unigram | 加一 | 待填写 | 待填写 | 待填写 | 待填写 |
| Bigram | 加一 | 待填写 | 待填写 | 待填写 | 待填写 |
| Trigram | 加一 | 待填写 | 待填写 | 待填写 | 待填写 |
| Bigram | 加 K，K=待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| 插值模型 | K与权重=待填写 | 待填写 | 待填写 | 待填写 | 待填写 |


In [ ]:
# 实践区：任务一 / 子任务4
# TODO：在相同训练文本和词表上完成模型对照表。
# TODO：另外记录未平滑模型的零概率与未见历史。


### 分析区：模型差异

请结合具体计数与得分回答：

1. 高阶模型是否更容易遇到未见历史？
2. 增加模型阶数是否一定得到更有用的结果？
3. K 值变化后，模型对已见和未见搭配的偏好如何变化？
4. 为什么句子得分较高不等于它在所有语境下都更正确？

回答：待填写。

---


## 子任务5：将句子评分用于缺词补全

### 1. 应用思路

给定含一个空位的句子和四个候选词：

```text
工作人员 正在 ____ 最新 通知
候选词：提交、发布、召开、参加
```

分别填入得到四个完整句子，再计算各自的得分。最终选择：

$$
\hat c=\arg\max_{c\in\mathcal C}\log P(S_c)
$$

其中，$\mathcal C$ 为候选词集合，$S_c$ 为填入候选词后的完整句子。候选均作为单个词元，因此同一道题的候选句长度相同，可以直接比较对数概率。

### 2. 必须理解：右侧信息如何参与判断

```text
工作人员 正在 ____ 最新 通知
工作人员 正在 ____ 实验 报告
```

两道题的左侧相同。只计算候选词在“工作人员 正在”之后出现的概率，会对两题给出相同的候选偏好；完整句子评分还包含候选之后的预测项。

**N-gram 的右侧作用范围是有限的。**候选位于位置 $j$ 时，它最多影响位置 $j$ 到 $j+N-1$ 的预测项，且不超过句末预测位置：

- Unigram 只改变候选本身的概率，不建模词序。
- Bigram 改变候选本身及紧随其后的一个词的预测概率。上例直接比较的是“候选 → 最新”或“候选 → 实验”；更后面的“最新 → 通知”“实验 → 报告”在各自题内是公共项。
- Trigram 还可能改变后面第二个词的预测概率，例如“候选 最新 → 通知”。
- 当空位在句末时，预测 `</s>` 的概率也可能随候选改变。

因此，完整句评分能够利用右侧的局部搭配，但并不等于能理解任意远处的语义。实际解释时，应列出随候选变化的条件概率项。

本子任务只明确应用思路。以上句子属于任务二的项目场景；不要用任务一的六句词表评价它们后误把大量 `<unk>` 当作正常项目结果。完整的数据读取、候选替换、排序和评价程序在任务二中独立编写。

### 分析区：从评分到补全

- 只根据空位左侧评分有什么局限：待填写。
- 对第一个例句，写出 Bigram 中随候选改变的条件概率项：待填写。
- 写出 Trigram 相比 Bigram 额外利用的一项右侧概率：待填写。
- 空位位于句末时，为什么不能省略 `</s>`：待填写。

---


# 四、任务二：基于统计语言模型的项目实操


## 项目名称

**基于统计语言模型的中文句子缺词补全**

本任务仅提供项目需求、统一数据和验收要求。请独立完成代码，可以复用任务一中自己编写的函数。


## 一、项目背景与现实需求

### 1. 项目背景

设想一个面向中文学习者的在线阅读练习平台。平台提供“根据上下文选择合适词语”的练习，例如：

> 工作人员正在 ______ 最新通知。  
> A．提交　B．发布　C．召开　D．参加

学习者需要结合上下文和词语搭配判断。例如，“发布通知”“提交报告”“召开会议”分别对应不同的表达习惯。

本项目要求开发一个小型程序：利用训练文本中的统计信息，对给定题目的候选词排序，并展示判断依据。

### 2. 现实需求

| 需求 | 程序应提供的功能 |
|---|---|
| 学习者需要选词建议 | 输出推荐词和候选排序 |
| 使用者需要理解判断依据 | 展示影响排序的关键条件概率 |
| 新句子没有在训练文本中出现 | 利用已学到的搭配关系评分 |
| 未见词组可能导致零概率 | 使用平滑处理 |
| 部分题目存在多个合理答案 | 标记并列或不确定情况，保留人工分析空间 |
| 需要比较不同方法 | 在相同数据上比较模型设置 |

项目定位为**教学辅助原型**。模型得分反映当前语料中的统计倾向，不作为中文表达绝对正确与否的判定。

### 3. 项目范围

- 已经分词的中文文本；
- 每题一个空位；
- 每题四个给定候选词；
- 每个候选词作为一个词语单位；
- 基于 N-gram 的完整句子评分。

不要求自行生成任意候选词，也不要求开发中文分词工具。不得将“发布—通知”等搭配直接写成判断规则代替模型评分。


## 二、项目输入与输出

### 1. 输入

```text
题目编号：T01
句子：工作人员 正在 ____ 最新 通知
A：提交
B：发布
C：召开
D：参加
```

空位标记 `____` 是独立词元，不加入词表。替换时保留各候选词的原始文本，用于最终展示；映射后的 `<unk>` 序列仅用于模型评分及诊断。

### 2. 输出

| 输出内容 | 含义 |
|---|---|
| 题目编号 | 对应输入题目 |
| 预测选项 | A、B、C或D |
| 预测词语 | 推荐填入的原始词语 |
| 完整句子 | 用原始候选替换空位后的结果 |
| 四个候选得分 | 使用同一评分方式，保留足够数值精度 |
| 候选排序 | 按得分从高到低排列 |
| 是否并列 | 最高分是否由多个候选共享 |

每道题候选句长度相同，可以直接比较完整句子的对数概率。不同题目之间长度可能不同，不应直接把原始总得分当作跨题难度或正确性指标。

**并列与异常规则必须事先确定：**

- 最高分并列时，可按 A、B、C、D 顺序输出，同时单独记录并列选项。
- 浮点误差可能导致理论等分出现极小差异。应预先规定有限分数的比较容差，例如绝对差不超过 $10^{-10}$，不要先四舍五入再决定排序。
- `-inf` 与无法估计分别处理；四候选全为 `-inf` 时记录“全部零概率、无有效区分”，不要对无穷值直接相减判断并列；存在无法评分的候选时，应按事前规则记录题目状态。
- 不能查看参考答案后再决定如何处理并列或无法评分。主要实验使用正 K 平滑以得到可比较的有限分数。

---


## 三、需要完成的项目任务

### 子任务1：读取数据并检查数据结构

需要完成：

1. 读取训练文本、验证题和测试题，并读取挑战题供后续分析。
2. 将训练文本组织为按句划分的词语列表。
3. 统计句子数、总词数和不重复词语数，不含特殊标记。
4. 检查每题编号唯一、含一个空位和四个非空的单词候选。
5. 只根据训练文本建立词表，不利用验证、测试或挑战题扩充词表。
6. 确定词表外词的统一处理方式。

基础数据规模较小，保留所有训练词语，不删除低频词。训练文本为 UTF-8；CSV 为 UTF-8 with BOM，读取 CSV 时建议指定 `encoding="utf-8-sig"`，避免首列字段名残留 BOM。

**验收：**数据统计、字段检查、词表定义与 OOV（词表外词）处理说明。


In [ ]:
# 实践区：任务二 / 子任务1
# 请独立编写数据读取与检查代码。
# 在下方记录实际运行结果；也可调用自己编写的 .py 文件。


### 记录区：数据检查

| 项目 | 实际结果 |
|---|---|
| 训练句子数 | 待填写 |
| 训练普通词次 | 待填写 |
| 训练普通词种数 | 待填写 |
| 预测词表大小及构成 | 待填写 |
| 验证 / 测试 / 挑战题数 | 待填写 |
| 空位与选项检查 | 待填写 |
| 编号与答案字段检查 | 待填写 |
| OOV统一处理规则 | 待填写 |

### 子任务2：建立模型

必做：

- Unigram 模型；
- Bigram 模型；
- 加一平滑；
- 至少两组其他 K 值的加 K 平滑，例如 K=0.1、0.5。

提高任务：

- Trigram 模型；
- 插值模型。

所有模型使用相同的72句训练文本和预测词表。可以复用任务一的自编函数，但需重新基于项目训练语料建立统计量。

**验收：**模型定义、参数记录、边界处理与概率归一化检查。


In [ ]:
# 实践区：任务二 / 子任务2
# 请独立建立必做模型；提高模型按要求选做。


### 子任务3：实现缺词补全功能

需要完成：

- 用每个候选词分别替换空位；
- 对完整候选句评分，包含句末预测；
- 输出候选排序；
- 记录并列情况；
- 支持对一批题目自动运行；
- 保留关键条件概率，支持解释排序。

不得使用人工搭配规则、参考答案查表或随机赋分代替语言模型评分。

**验收：**批量补全结果与至少一题的四个候选完整评分记录。


In [ ]:
# 实践区：任务二 / 子任务3
# 请独立实现候选替换、完整句评分、排序、并列记录及批量输出。


### 记录区：先确定规则

- 最高分并列的判定方法及容差：待填写。
- 并列时的确定性选择规则：待填写。
- 零概率、未见历史和无法评分的记录方式：待填写。
- 输出文件格式与字段：待填写。

### 子任务4：利用验证集选择设置

比较不同模型和平滑参数，记录：

- 答对题数；
- 准确率；
- 最高分并列题数；
- 零概率或无法评分的题数。

主要选择依据为验证集准确率。准确率相同时，可以依次比较并列题数、模型复杂程度；如果仍然相同，再按事先规定的配置顺序选择。应在测试前明确规则。

为避免统计口径不一致，建议把“至少一个候选零概率的题数”“四候选全部零概率的题数”和“存在无法评分候选的题数”分别记录；表中简写时说明采用哪种口径。

选定配置后固定训练文本、词表、模型阶数、K、插值权重及并列规则，再运行测试集。不得根据测试答案返回修改参数。

**验收：**验证集对照表、配置选择依据与最终配置记录。


In [ ]:
# 实践区：任务二 / 子任务4
# 请独立完成验证集比较与配置选择，保存选定设置。


### 记录区：验证集模型对比

| 模型 | 平滑设置 | 答对数/12 | 准确率 | 并列题数 | 零概率题数及口径 | 无法评分题数 | 最终选用 |
|---|---|---:|---:|---:|---|---:|---|
| Unigram | 加一 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| Bigram | 加一 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| Bigram | K=待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| Bigram | K=待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| 提高模型（选做） | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |

**测试前冻结配置：**

| 项目 | 事先记录 |
|---|---|
| 最终模型 / 阶数 | 待填写 |
| K值 / 各分量参数 | 待填写 |
| 插值权重（如使用） | 待填写 |
| 训练文本与词表版本 | 待填写 |
| OOV与句子边界处理 | 待填写 |
| 分数比较与并列规则 | 待填写 |
| 验证集同分时的模型选择规则 | 待填写 |
| 选择依据 | 待填写 |
| 配置确定时间 | 待填写 |

### 子任务5：测试与结果分析

需要完成：

1. 使用已固定配置，批量输出测试题预测结果。
2. 保存预测结果，读取 `data/test.csv` 中的 `answer` 列，按题目编号对齐并直接计算：

$$
\text{准确率}=\frac{\text{预测选项与参考答案一致的题数}}{\text{测试题总数}}
$$

3. 单独报告最高分并列题数；并列时仍按事先规定的选择规则评价，不能临时改为“参考答案在并列集合中就算对”。
4. 选择至少三道题，说明影响排序的关键条件概率。
5. 分析错误题或无法区分的题目。

如果全部答对，应分析题目难度和语料特点，不能只给出“模型效果很好”的结论。参考答案已随数据提供，仅用于评价；不得用参考答案生成预测或重新调整已确定的模型配置。

**验收：**测试预测文件、准确率评价、关键概率分析与局限性说明。


In [ ]:
# 实践区：任务二 / 子任务5
# 请独立批量运行测试并保存结果。
# 使用 test.csv 的 answer 列补充参考答案、是否命中及准确率。


### 分析区：测试结果

- 最终模型及参数：待填写。
- 预测文件路径：待填写。
- 答对数与准确率：与 test.csv 的 answer 列比较后填写。
- 最高分并列题数及编号：待填写。
- 零概率与无法评分情况：待填写。

| 题号 | 对比候选 | 关键历史 → 预测词 | 计数与条件概率 | 如何影响排序 |
|---|---|---|---|---|
| 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |

错误、并列或全部答对时的数据难度分析：待填写。

### 子任务6：完成挑战题与独立编题

需要完成：

- 分析下文六道挑战题；
- 自己编写至少六道新题，每题一个空位、四个单词候选；
- **先记录参考答案或可接受答案及理由，再运行模型**，保留记录时间；
- 至少包含一组“左侧相同、右侧不同”的题；
- 至少包含一道存在多个合理答案的题。

自编题用于观察已经确定的模型，不根据其答案反复修改训练数据。挑战题和自编题与普通测试分开报告。

**验收：**六道挑战题分析、六道自编题及预先判断、固定模型的输出和解释。


In [ ]:
# 实践区：任务二 / 子任务6
# 请先在下方记录自编题与参考判断，再运行固定模型。
# 自行完成挑战题、自编题的批量处理与分析。


### 记录区：自编题（先填写，再运行）

| 编号 | 缺词句 | A | B | C | D | 预先可接受选项 | 理由 | 记录时间 |
|---|---|---|---|---|---|---|---|---|
| S01 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| S02 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| S03 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| S04 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| S05 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| S06 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |

- 左侧相同、右侧不同的题号：待填写。
- 存在多个合理答案的题号：待填写。

| 编号 | 模型预测 | 四个得分 / 排序 | 并列情况 | 与预先判断比较 | 原因分析 |
|---|---|---|---|---|---|
| S01 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| S02 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| S03 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| S04 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| S05 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| S06 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |

---


## 四、实验数据说明

### 1. 数据构成

本实验提供一套自编教学数据：

| 数据部分 | 数量 | 用途 |
|---|---:|---|
| 训练文本 | 72句 | 统计词频与N-gram |
| 验证题 | 12题 | 比较模型与选择参数 |
| 测试题 | 12题 | 固定配置后的评价 |
| 挑战题 | 6题 | 分析歧义、未登录词和句子边界 |

这些材料专门用于观察统计语言模型的行为，不是公开基准或真实用户日志。

### 2. 数据设计特点

- 12个主要候选词在训练中各出现6次：发布、提交、召开、参加、安装、排查、遵守、保护、预测、计算、扩大、降低。
- 验证和测试的参考完整句没有直接重复训练原句。
- 普通验证题和测试题中的词语均在训练词表中出现过。
- 测试集包含三组“相同左侧、不同右侧”的题目。
- 各集合共享部分句式和词组，属于教学性组合测试。

不能把该小数据集上的高准确率解释为广泛的中文理解能力。

### 3. 数据文件与字段

训练文本 `data/train.txt` 每行一句，无分组标题。题目文件如下：

| 文件 | 字段 |
|---|---|
| `data/dev.csv` | `id,sentence,option_A,option_B,option_C,option_D,answer` |
| `data/test.csv` | `id,sentence,option_A,option_B,option_C,option_D,answer` |
| `data/challenge.csv` | `id,sentence,option_A,option_B,option_C,option_D,analysis_focus` |

| 字段 | 含义 |
|---|---|
| `id` | 题目编号 |
| `sentence` | 含一个独立 `____` 词元的分词句子 |
| `option_A`—`option_D` | 四个候选词，每个为一个词语单位 |
| `answer` | 参考选项，在验证集和测试集中均提供；用于评价，不参与预测 |
| `analysis_focus` | 挑战题分析重点，不是参考答案 |

**验证集和测试集均包含参考答案，可直接运行准确率验证。测试集答案仅用于固定配置后的评价，不用于调参。**挑战题不设置统一的单参考答案列。

---


## 五、训练文本：72句完整语料

每行是一个独立句子。下列分组标题不属于训练文本，三组文本需要合并用于训练；配套 `train.txt` 已合并好。

### 第一组：通知、材料与活动

```text
工作人员 正在 发布 通知
学校 今天 发布 最新 通知
平台 及时 发布 比赛 结果
公司 明天 发布 产品 信息
网站 已经 发布 招生 公告
部门 定期 发布 工作 安排
工作人员 正在 提交 材料
学生 今天 提交 实验 报告
申请人 按时 提交 入学 申请
教师 明天 提交 教学 方案
作者 已经 提交 修改 稿件
团队 需要 提交 项目 计划
学校 今天 召开 工作 会议
公司 明天 召开 新闻 发布会
部门 定期 召开 例会
会议 将于 明天 召开
研讨会 已经 顺利 召开
团队 准备 召开 项目 讨论会
学生 今天 参加 学术 讲座
同学 明天 参加 技能 比赛
教师 定期 参加 教学 培训
工作人员 需要 参加 工作 会议
申请人 必须 参加 入学 考试
团队 已经 参加 项目 展示
```


### 第二组：设备、规则与安全

```text
技术人员 正在 安装 软件
用户 今天 安装 新 的 软件
工程师 明天 安装 测试 设备
工作人员 已经 安装 安全 系统
学生 需要 安装 实验 程序
团队 准备 安装 新 的 设备
技术人员 正在 排查 故障
工程师 今天 排查 设备 故障
工作人员 定期 排查 安全 隐患
团队 正在 排查 系统 问题
学生 需要 排查 程序 错误
部门 已经 排查 网络 故障
学生 必须 遵守 考试 纪律
工作人员 需要 遵守 安全 规定
用户 应当 遵守 使用 规则
团队 始终 遵守 工作 纪律
申请人 必须 遵守 考试 规则
教师 带头 遵守 学校 规定
平台 需要 保护 用户 隐私
工作人员 应当 保护 重要 数据
学生 自觉 保护 校园 环境
公司 始终 保护 用户 信息
团队 共同 保护 自然 资源
学校 高度 重视 保护 学生 隐私
```


### 第三组：预测、计算与变化

```text
研究人员 正在 预测 天气 变化
模型 可以 预测 市场 需求
团队 今天 预测 产品 销量
专家 正在 预测 经济 走势
工作人员 需要 预测 明天 的 天气
系统 能够 预测 客流 变化
研究人员 正在 计算 概率
学生 今天 计算 模型 得分
程序 可以 计算 平均值
工程师 正在 计算 设备 成本
教师 需要 计算 考试 总分
系统 能够 计算 两地 距离
学校 计划 扩大 招生 规模
公司 准备 扩大 产品 市场
平台 需要 扩大 服务 范围
团队 正在 扩大 实验 规模
项目 将会 扩大 研究 范围
网站 逐步 扩大 信息 覆盖面
这种 方法 可以 降低 测量 误差
公司 需要 降低 设备 成本
工程师 正在 降低 环境 噪声
平台 持续 降低 使用 风险
工作人员 努力 降低 能源 损耗
新 的 系统 能够 降低 工作 强度
```

按上述分词方式，不包含句首、句尾等特殊标记时，共有 **358个词次、141种词语**。若保留所有训练词并添加 `</s>` 和 `<unk>`，预测词表大小应为 **143**。

“发布会”是一个完整词，不计入“发布”的出现次数。请按词语统计，不能按字符串子串计数。


## 六、验证题

验证集用于比较模型和选择参数，包含参考答案。对应文件：`data/dev.csv`。

| 编号 | 缺词句 | A | B | C | D | 参考答案 |
| --- | --- | --- | --- | --- | --- | --- |
| D01 | 网站 已经 ____ 最新 通知 | 发布 | 提交 | 参加 | 召开 | A |
| D02 | 学生 明天 ____ 实验 报告 | 参加 | 召开 | 发布 | 提交 | D |
| D03 | 工作 会议 将于 明天 ____ | 召开 | 提交 | 发布 | 参加 | A |
| D04 | 同学 今天 ____ 入学 考试 | 召开 | 发布 | 参加 | 提交 | C |
| D05 | 用户 明天 ____ 新 的 软件 | 安装 | 遵守 | 保护 | 排查 | A |
| D06 | 工程师 正在 ____ 网络 故障 | 遵守 | 排查 | 保护 | 安装 | B |
| D07 | 学生 应当 ____ 考试 规则 | 安装 | 排查 | 遵守 | 保护 | C |
| D08 | 平台 始终 ____ 用户 隐私 | 排查 | 安装 | 遵守 | 保护 | D |
| D09 | 模型 能够 ____ 天气 变化 | 计算 | 预测 | 扩大 | 降低 | B |
| D10 | 教师 今天 ____ 考试 总分 | 降低 | 扩大 | 预测 | 计算 | D |
| D11 | 平台 计划 ____ 服务 范围 | 降低 | 扩大 | 预测 | 计算 | B |
| D12 | 这种 方法 能够 ____ 环境 噪声 | 预测 | 扩大 | 降低 | 计算 | C |

参考答案用于统一比较模型，不表示其他表达在所有语境下都不成立。认为存在其他合理答案时，应单独记录理由。


## 七、测试题

测试集提供题目、选项及参考答案。模型设置确定后再完成测试，保存预测结果，并直接与参考答案比较计算准确率。对应文件：`data/test.csv`。

| 编号 | 缺词句 | A | B | C | D | 参考答案 |
| --- | --- | --- | --- | --- | --- | --- |
| T01 | 工作人员 正在 ____ 最新 通知 | 提交 | 发布 | 召开 | 参加 | B |
| T02 | 工作人员 正在 ____ 实验 报告 | 参加 | 召开 | 发布 | 提交 | D |
| T03 | 项目 讨论会 将于 明天 ____ | 发布 | 参加 | 召开 | 提交 | C |
| T04 | 申请人 今天 ____ 技能 比赛 | 召开 | 参加 | 发布 | 提交 | B |
| T05 | 技术人员 正在 ____ 新 的 软件 | 安装 | 排查 | 保护 | 遵守 | A |
| T06 | 技术人员 正在 ____ 设备 故障 | 遵守 | 保护 | 排查 | 安装 | C |
| T07 | 工作人员 必须 ____ 安全 规定 | 排查 | 保护 | 安装 | 遵守 | D |
| T08 | 公司 应当 ____ 重要 数据 | 遵守 | 排查 | 安装 | 保护 | D |
| T09 | 研究人员 正在 ____ 市场 需求 | 降低 | 预测 | 计算 | 扩大 | B |
| T10 | 研究人员 正在 ____ 模型 得分 | 计算 | 降低 | 扩大 | 预测 | A |
| T11 | 学校 准备 ____ 研究 范围 | 扩大 | 计算 | 降低 | 预测 | A |
| T12 | 新 的 方法 可以 ____ 测量 误差 | 预测 | 计算 | 降低 | 扩大 | C |

### 重点分析题组

- **T01与T02：**相同左侧，不同的“通知”和“报告”语境。
- **T05与T06：**相同左侧，不同的软件与故障语境。
- **T09与T10：**相同左侧，不同的市场需求与模型得分语境。

解释完整句评分如何利用右侧词语，而不只是报告预测答案。比较时按候选词本身对齐，注意各题的候选选项顺序不同。列出各阶模型实际变化的概率项，并说明哪些后续项在同题四个候选之间相同。

### 分析区：三组对照

| 对照题组 | 左侧是否相同 | 右侧差异 | 随候选变化的概率项 | 对排序的影响 |
|---|---|---|---|---|
| T01 / T02 | 待填写 | 待填写 | 待填写 | 待填写 |
| T05 / T06 | 待填写 | 待填写 | 待填写 | 待填写 |
| T09 / T10 | 待填写 | 待填写 | 待填写 | 待填写 |


## 八、挑战题

挑战题不与普通测试题合并计算单参考答案准确率。对应文件：`data/challenge.csv`。下面 A—D 的顺序与文件中的选项顺序一致。

| 编号 | 缺词句 | A | B | C | D | 分析重点 |
| --- | --- | --- | --- | --- | --- | --- |
| C01 | 团队 明天 ____ 工作 会议 | 召开 | 参加 | 发布 | 提交 | “召开”和“参加”可能都成立，主体角色未说明 |
| C02 | 公司 需要 ____ 设备 成本 | 计算 | 降低 | 扩大 | 预测 | 不同任务背景下可能有不同合理答案 |
| C03 | 技术人员 正在 ____ 量子芯片 | 安装 | 排查 | 遵守 | 保护 | 右侧包含训练中未出现的词 |
| C04 | 研究人员 正在 ____ 模型 得分 | 评估 | 核验 | 汇总 | 审查 | 四个候选词均未出现在训练词表中 |
| C05 | ____ 应当 遵守 考试 规则 | 申请人 | 学生 | 用户 | 工作人员 | 空位位于句首，且可能有多个合理答案 |
| C06 | 新闻 发布会 将于 明天 ____ | 召开 | 参加 | 发布 | 提交 | 空位位于句末，需要正确处理句子结束 |

每道题需要分析：

1. 程序能否正常评分？
2. 哪些词被映射为未登录词标记？
3. 是否出现并列？
4. 模型推荐与人工判断是否一致？
5. 差异来自语料、模型，还是题目本身的歧义？

对于 C04，如果四个候选都映射为同一个 `<unk>`，映射后的候选句相同，确定性评分应给出相同结果。应如实报告，不能通过随机赋分制造差异。仍需保留四个原始候选词用于展示。

### 分析区：挑战题

| 题号 | 可否评分 / 得分 | OOV词与映射 | 排序及并列 | 人工可接受答案与理由 | 差异来源 |
|---|---|---|---|---|---|
| C01 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| C02 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| C03 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| C04 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| C05 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
| C06 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |

---


# 五、实验结果与提交内容


## 1. 任务一结果

提交：

- Unigram、Bigram、Trigram 统计结果；
- 指定条件概率查询结果，含计数与分母；
- 至少一个句子的逐项概率与总得分，含句末预测；
- 不同平滑方法的对比与概率归一化检查；
- 模型阶数与句子得分的分析；
- 对零概率、未见历史和修改语料前后变化的说明。


## 2. 任务二模型对比表

请完成前面的验证集模型对比表，至少包含以下内容；需要时增加参数行：

| 模型 | 平滑设置 | 验证集答对数 | 并列题数 | 是否选为最终模型 |
|---|---|---:|---:|---|
| Unigram | 待填写 | 待填写 | 待填写 | 待填写 |
| Bigram | 加一 | 待填写 | 待填写 | 待填写 |
| Bigram | 第一组其他K值 | 待填写 | 待填写 | 待填写 |
| Bigram | 第二组其他K值 | 待填写 | 待填写 | 待填写 |
| 提高模型（选做） | 待填写 | 待填写 | 待填写 | 待填写 |

报告中可引用前面的完整表格，无需重复计算。


## 3. 逐题预测文件

推荐使用 CSV，至少包含：

```text
id                  题目编号
predicted_option    预测选项
predicted_word      预测词语（保留原词）
completed_sentence  补全后的原始句子
score_A             A的完整句自然对数得分
score_B             B的完整句自然对数得分
score_C             C的完整句自然对数得分
score_D             D的完整句自然对数得分
ranking             候选排序
is_tie              最高分是否并列
```

建议另加 `tied_options` 和 `status`，如实记录并列选项、零概率及无法评分。所有记录使用一致的格式；非有限得分和缺失值如何保存，应在报告中说明。从 `test.csv` 的 `answer` 列读取参考答案，补充 `reference_answer` 和 `is_correct`，并计算准确率。


## 4. 独立实践材料

提交：

- 完成后的 Notebook 或自己编写的项目代码；
- 至少六道自编题及事先记录的参考判断；
- 六道挑战题分析；
- 关键结果与问题说明。

建议命名与组织：

```text
学号_姓名_实验3/
├── NLP实验3_统计语言模型_学号_姓名.ipynb
├── src/                         # 如使用独立Python脚本
├── results/
│   ├── model_comparison.csv
│   ├── test_predictions.csv
│   ├── challenge_analysis.md
│   └── self_questions.csv
└── 实验报告.pdf
```


## 5. 实验报告中的核心问题

报告应回答：

1. Unigram 为什么容易对候选词给出相同得分？结合12个主要候选词的训练频次说明。
2. Bigram 在哪些题目上利用了空位右侧的信息？它能直接利用多远的右侧上下文？
3. 平滑怎样影响未见搭配和候选排序？
4. 为什么高阶模型不一定更适合小语料？
5. 为什么有些题目不能简单判定为“模型正确”或“模型错误”？
6. 这套72句教学语料的结果能够说明什么，又不能说明什么？

### 分析区：报告核心回答

1. Unigram 与等频候选：待填写。
2. Bigram 与右侧信息：待填写。
3. 平滑与排序变化：待填写。
4. 模型阶数与数据稀疏：待填写。
5. 歧义、OOV与评价局限：待填写。
6. 教学组合测试的结论边界：待填写。

---


# 六、实验总结

- 本次实验掌握的关键方法。
- 最有代表性的一个结果及其计数依据。
- 遇到的问题、定位过程与解决办法。
- 如果扩展语料或改进模型，计划如何设计新的验证。

**根据项目需求和统一数据，独立完成代码实现、模型运行、结果整理和解释。**
